# Agent & LLM Evaluation: MLflow Prompt Registry and LLM-as-Judge

This notebook demonstrates how to evaluate the Custom Deep Research pipeline using
**MLflow Prompt Registry** for versioned evaluation prompts and **LLM-as-Judge** for
automated quality scoring. You will register scoring prompts, run the pipeline against
a test dataset, evaluate results with `mlflow.evaluate()`, and compare metrics across
experiment runs.

## 1. Load Environment

In [ ]:
import os
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

mlflow_uri = os.getenv("MLFLOW_TRACKING_URI", "")
experiment = os.getenv("MLFLOW_EXPERIMENT_NAME", "deep-research-harness")

import mlflow
mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment(experiment)
exp = mlflow.get_experiment_by_name(experiment)

print(f"MLflow Tracking URI: {mlflow_uri or '(not set)'}")
print(f"Experiment: {experiment}")
if exp:
    print(f"\u2705 Connected to experiment: {exp.name} (ID: {exp.experiment_id})")
else:
    print("\u26a0\ufe0f Experiment not found \u2014 it will be created on first run")

## 2. Register Evaluation Prompts in MLflow Prompt Registry

In [ ]:
import mlflow

quality_judge_prompt = mlflow.register_prompt(
    name="quality_judge_prompt",
    template="""You are a research quality evaluator. Score the following research report
on a scale of 1-10 across these dimensions:
- Relevance: Does the report address the query?
- Completeness: Are all aspects of the query covered?
- Accuracy: Are claims supported by citations?
- Coherence: Is the report well-structured and readable?

Query: {{query}}
Report: {{report}}

Return a JSON object with scores for each dimension and an overall score:
{"relevance": <1-10>, "completeness": <1-10>, "accuracy": <1-10>, "coherence": <1-10>, "overall": <1-10>, "reasoning": "<text>"}""",
)
print(f"\u2705 Registered prompt: {quality_judge_prompt.name} v{quality_judge_prompt.version}")

## 3. Load Test Dataset

In [ ]:
import json

dataset_path = os.path.join(
    os.path.dirname(os.getcwd()), "config", "eval", "datasets", "test_queries.jsonl"
)

test_queries = []
if os.path.exists(dataset_path):
    with open(dataset_path) as f:
        for line in f:
            if line.strip():
                test_queries.append(json.loads(line))
    print(f"Loaded {len(test_queries)} test queries")
    for i, q in enumerate(test_queries[:5], 1):
        print(f"  {i}. {q['query']}")
else:
    print(f"\u26a0\ufe0f Dataset not found at: {dataset_path}")
    print("  Using built-in fallback queries")
    test_queries = [
        {"query": "What is the main architecture described in the document?"},
        {"query": "What are the key contributions and innovations?"},
        {"query": "What evaluation metrics are used and what are the results?"},
    ]
    for i, q in enumerate(test_queries, 1):
        print(f"  {i}. {q['query']}")
print("\u2705 Dataset loaded")

## 4. Run Research Pipeline on Test Queries

In [ ]:
import time
import httpx

BACKEND_URL = f"http://localhost:{os.getenv('BACKEND_PORT', '8000')}"
results = []

print(f"Running {len(test_queries)} queries against {BACKEND_URL}\n")

for i, item in enumerate(test_queries, 1):
    query = item["query"]
    print(f"  [{i}/{len(test_queries)}] {query[:60]}...", end=" ")
    start = time.time()
    try:
        resp = httpx.post(
            f"{BACKEND_URL}/research",
            json={"query": query},
            timeout=180,
        )
        latency = time.time() - start
        if resp.status_code == 200:
            data = resp.json()
            results.append({
                "query": query,
                "response": data.get("report", data.get("result", "")),
                "latency": latency,
                "success": True,
            })
            print(f"\u2705 ({latency:.1f}s)")
        else:
            results.append({"query": query, "response": "", "latency": latency, "success": False})
            print(f"\u274c HTTP {resp.status_code}")
    except Exception as e:
        latency = time.time() - start
        results.append({"query": query, "response": "", "latency": latency, "success": False})
        print(f"\u274c {e}")

success_count = sum(1 for r in results if r["success"])
print(f"\n\u2705 Pipeline run complete: {success_count}/{len(results)} succeeded")

## 5. Run LLM-as-Judge Evaluation

In [ ]:
import mlflow
import pandas as pd

successful_results = [r for r in results if r["success"]]

if not successful_results:
    print("\u26a0\ufe0f No successful results to evaluate \u2014 run the pipeline first")
else:
    eval_data = pd.DataFrame({
        "inputs": [r["query"] for r in successful_results],
        "predictions": [r["response"] for r in successful_results],
    })

    with mlflow.start_run(run_name="llm-as-judge-eval"):
        eval_results = mlflow.evaluate(
            data=eval_data,
            model_type="question-answering",
            evaluators="default",
        )
        print("\u2705 Evaluation complete")
        print(f"\nMetrics:")
        for k, v in eval_results.metrics.items():
            print(f"  {k}: {v}")

## 6. Compare Across Experiments

In [ ]:
import mlflow

if exp:
    runs = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string='tags.mlflow.runName = "llm-as-judge-eval"',
        order_by=["start_time DESC"],
        max_results=10,
    )
    if len(runs) > 0:
        print(f"Found {len(runs)} evaluation runs\n")
        display_cols = [c for c in runs.columns if c.startswith("metrics.")]
        if display_cols:
            comparison = runs[["start_time"] + display_cols].head(5)
            print(comparison.to_string(index=False))
        else:
            print("  No metric columns found yet")
    else:
        print("No evaluation runs found \u2014 run section 5 first")
else:
    print("\u26a0\ufe0f No experiment configured")

print("\n\u2705 Comparison complete")

## 7. Summary

In [ ]:
print("\u2705 LLM-as-Judge evaluation walkthrough complete")
print()
print("What we covered:")
print("  1. Registered versioned evaluation prompts in MLflow Prompt Registry")
print("  2. Loaded test dataset for reproducible evaluation")
print("  3. Ran the research pipeline on test queries")
print("  4. Evaluated outputs with mlflow.evaluate() (LLM-as-Judge)")
print("  5. Compared metrics across experiment runs")
print()
print("Architecture:")
print("  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510    \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510    \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510")
print("  \u2502 Test Dataset     \u2502 \u2192  \u2502 Research Pipeline    \u2502 \u2192  \u2502 LLM-as-Judge     \u2502")
print("  \u2502 (test_queries)   \u2502    \u2502 (backend /research)  \u2502    \u2502 (mlflow.evaluate)\u2502")
print("  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518    \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518    \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518")
print("                                         \u2502")
print("                                         \u25bc")
print("                              \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510")
print("                              \u2502 MLflow Experiment  \u2502")
print("                              \u2502 (metrics + traces) \u2502")
print("                              \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518")